In [1]:
import os
import json
import uproot
import awkward as ak
import numpy as np

In [2]:
BASE_PATH = "/home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root"
LUMI = 36100.0  # in pb^-1 (36.1 fb^-1)
REDUCTION = 0.6 # You processed 60% of total files

BRANCHES = [
    # --- Global Event & MET ---
    "MET_Core_AnalysisMETAuxDyn_mpx",
    "MET_Core_AnalysisMETAuxDyn_mpy",
    "MET_Core_AnalysisMETAuxDyn_sumet",
    "EventInfoAuxDyn_mcEventWeights",
    "EventInfoAuxDyn_DFCommonJets_eventClean_LooseBad", # Required for jet cleaning
    "EventInfoAuxDyn_runNumber", 
    
    # --- Small-R Jets (Selection & Cleaning) ---
    "AnalysisJetsAuxDyn_pt",
    "AnalysisJetsAuxDyn_eta",
    "AnalysisJetsAuxDyn_phi",
    "AnalysisJetsAuxDyn_NNJvtPass",
    "AnalysisJetsAuxDyn_SumPtTrkPt500",    # Required for f_ch cleaning
    "AnalysisJetsAuxDyn_EnergyPerSampling", # Required for f_max cleaning
    "AnalysisJetsAuxDyn_NumTrkPt500",       # Required for Overlap Removal logic
    
    # --- Large-R Jets (AD Features - Expanded) ---
    "AnalysisLargeRJetsAuxDyn_pt",
    "AnalysisLargeRJetsAuxDyn_eta",
    "AnalysisLargeRJetsAuxDyn_phi",
    "AnalysisLargeRJetsAuxDyn_m",
    "AnalysisLargeRJetsAuxDyn_Tau1_wta",
    "AnalysisLargeRJetsAuxDyn_Tau2_wta",
    "AnalysisLargeRJetsAuxDyn_Tau3_wta",
    "AnalysisLargeRJetsAuxDyn_D2", # Highly recommended for AD
    "AnalysisLargeRJetsAuxDyn_C2", # Highly recommended for AD
    
    # --- Lepton & Tau Vetoes ---
    "AnalysisElectronsAuxDyn_pt",
    "AnalysisElectronsAuxDyn_eta",
    "AnalysisElectronsAuxDyn_phi",
    "AnalysisElectronsAuxDyn_DFCommonElectronsLHTight",
    "AnalysisMuonsAuxDyn_pt",
    "AnalysisMuonsAuxDyn_eta",
    "AnalysisMuonsAuxDyn_phi",
    "AnalysisMuonsAuxDyn_muonType",
    "AnalysisMuonsAuxDyn_quality",
    "AnalysisTauJetsAuxDyn_pt",
    "AnalysisTauJetsAuxDyn_eta",
    "AnalysisTauJetsAuxDyn_phi",
    "AnalysisTauJetsAuxDyn_JetDeepSetTight",
    
    # --- Flavor Tagging ---
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu",
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc",
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"
]


In [3]:
import os
import uproot
import awkward as ak
import dask_awkward as dak

def load_data_events_lazy(split_type, branches, periods=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'K', 'L']):
    all_runs_lazy = []

    for period in periods:
        # Note: Added 'reduce_root' to path based on your error logs
        period_dir = os.path.join(BASE_PATH, period) 
        if not os.path.exists(period_dir):
            continue

        run_dirs = [d for d in os.listdir(period_dir) if d.startswith("run")]
        
        for run_name in run_dirs:
            run_path = os.path.join(period_dir, run_name)
            files = sorted([os.path.join(run_path, f) for f in os.listdir(run_path) if f.endswith(".root")])
            
            # --- Robust File Validation ---
            valid_files = []
            for f in files:
                try:
                    with uproot.open(f) as temp_f:
                        if "CollectionTree" in temp_f:
                            valid_files.append(f)
                except Exception:
                    continue # Ignore corrupted files/header errors
            
            n = len(valid_files)
            if n == 0: continue

            # Slicing logic (40/20/20/20)
            i1, i2, i3 = int(0.4 * n), int(0.6 * n), int(0.8 * n)
            if split_type == "train": selected = valid_files[:i1]
            elif split_type == "val": selected = valid_files[i1:i2]
            elif split_type == "ad": selected = valid_files[i2:i3]
            else: selected = valid_files[i3:]

            if not selected: continue

            try:
                # --- NEW SYNTAX: List of files + key ---
                # This treats all selected files for this run as a single virtual array
                events = uproot.dask(
                    selected, 
                    key="CollectionTree", 
                    filter_name=branches
                )
                
                # Add weight branch lazily
                if "EventInfoAuxDyn_mcEventWeights" in branches:
                    # mcEventWeights is usually a vector; take the first element
                    events["weight_phys"] = events["EventInfoAuxDyn_mcEventWeights"][:, 0]
                else:
                    # For real Data, weights are just 1.0. 
                    # Use dak.ones_like to create a lazy column of 1s matching the length
                    events["weight_phys"] = dak.ones_like(events[branches[0]], dtype="float32")
                
                all_runs_lazy.append(events)
                print(f"Registered {len(selected)} files for {run_name} (Lazy)")

            except Exception as e:
                print(f"Skipping run {run_name} due to error: {e}")

    if not all_runs_lazy:
        print("No files were successfully registered.")
        return None

    # Concatenate all lazy arrays into one giant virtual array
    return dak.concatenate(all_runs_lazy)

# Usage
lazy_events = load_data_events_lazy("train", branches=BRANCHES)

Registered 226 files for run299584 (Lazy)
Registered 8 files for run299340 (Lazy)
Registered 40 files for run298687 (Lazy)
Registered 7 files for run299315 (Lazy)
Registered 42 files for run298595 (Lazy)
Registered 14 files for run298771 (Lazy)
Registered 44 files for run299055 (Lazy)
Registered 106 files for run299184 (Lazy)
Registered 29 files for run299147 (Lazy)
Registered 43 files for run297730 (Lazy)
Registered 54 files for run298862 (Lazy)
Registered 3 files for run299390 (Lazy)
Registered 35 files for run299288 (Lazy)
Registered 14 files for run299343 (Lazy)
Registered 27 files for run298633 (Lazy)
Registered 15 files for run298609 (Lazy)
Registered 234 files for run298967 (Lazy)
Registered 33 files for run299144 (Lazy)
Registered 13 files for run298773 (Lazy)
Registered 10 files for run299243 (Lazy)
Registered 6 files for run298690 (Lazy)
Registered 8 files for run299241 (Lazy)
Registered 91 files for run300908 (Lazy)
Registered 119 files for run300687 (Lazy)
Registered 56 fil

In [4]:
print("Lazy Dask-Awkward array structure:")

Lazy Dask-Awkward array structure:
